<a href="https://colab.research.google.com/github/ris2002/Multik30-Eng-De-_RNN_LSTM/blob/main/RNN_LSTM_Translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
from datasets import load_dataset
dataset = load_dataset("bentrevett/multi30k")
train_data=dataset['train']
test_data=dataset['test']
valid_data=dataset['validation']

In [45]:
len(train_data)

29000

In [46]:
len(test_data)

1000

In [47]:
len(valid_data)

1014

Data Checking

In [48]:
for item in train_data:
  print(item['en'])
  print(item['de'])
  break



Two young, White males are outside near many bushes.
Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.




> Data Cleaning



In [49]:
import re
def clean_data(text,lang):
  text=text.lower().strip()
  if lang=='en':
    text = re.sub(r"[^a-zA-Z\s]", "", text)
  elif lang=='de':
    text=re.sub(r"[^a-zA-ZäöüßÄÖÜ\s]", "", text)
  text = re.sub(r"\s+", " ", text) #cleans extra white space
  return text






In [50]:
cleaned_results = []
for item in train_data:
  en_cleaned=clean_data(item['en'], 'en')
  de_cleaned=clean_data(item['de'],'de')
  cleaned_results.append({'en': en_cleaned, 'de': de_cleaned})


In [51]:
for item in cleaned_results[:5]:
    print(f"EN: {item['en']}")
    print(f"DE: {item['de']}")
    print("-" * 10)

EN: two young white males are outside near many bushes
DE: zwei junge weiße männer sind im freien in der nähe vieler büsche
----------
EN: several men in hard hats are operating a giant pulley system
DE: mehrere männer mit schutzhelmen bedienen ein antriebsradsystem
----------
EN: a little girl climbing into a wooden playhouse
DE: ein kleines mädchen klettert in ein spielhaus aus holz
----------
EN: a man in a blue shirt is standing on a ladder cleaning a window
DE: ein mann in einem blauen hemd steht auf einer leiter und putzt ein fenster
----------
EN: two men are at the stove preparing food
DE: zwei männer stehen am herd und bereiten essen zu
----------


In [52]:
cleaned_results[-1]


{'en': 'a man in shorts and a hawaiian shirt leans over the rail of a pilot boat with fog and mountains in the background',
 'de': 'ein mann in shorts und hawaiihemd lehnt sich über das geländer eines lotsenboots mit nebel und bergen im hintergrund'}

Tokenisation Pipeline


In [53]:
!python -m spacy download de_core_news_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 89.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [54]:
import spacy
from spacy.lang.en import English
from spacy.lang.de import German
nlp_en=English()
nlp_de=German()
def tokenize_eng(cleaned_results:list,en_tokens:list):

  tokenizer=nlp_en.tokenizer
  for text in cleaned_results:
    token_list=[]
    eng_text=text['en']

    doc=tokenizer(eng_text)
    for token in doc:
      token_list.append(token.text)
    en_tokens.append(token_list)

def tokenize_de(cleaned_results:list,de_tokens:list):

  tokenizer=nlp_de.tokenizer
  for text in cleaned_results:
    token_list=[]
    de_text=text['de']

    doc=tokenizer(de_text)
    for token in doc:
      token_list.append(token.text)
    de_tokens.append(token_list)







In [55]:
en_tokens=[]
de_tokens=[]

In [56]:
tokenize_eng(cleaned_results,en_tokens)
tokenize_de(cleaned_results,de_tokens)

In [57]:
en_tokens[:5]

[['two',
  'young',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes'],
 ['several',
  'men',
  'in',
  'hard',
  'hats',
  'are',
  'operating',
  'a',
  'giant',
  'pulley',
  'system'],
 ['a', 'little', 'girl', 'climbing', 'into', 'a', 'wooden', 'playhouse'],
 ['a',
  'man',
  'in',
  'a',
  'blue',
  'shirt',
  'is',
  'standing',
  'on',
  'a',
  'ladder',
  'cleaning',
  'a',
  'window'],
 ['two', 'men', 'are', 'at', 'the', 'stove', 'preparing', 'food']]

In [58]:
de_tokens[:5]

[['zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche'],
 ['mehrere',
  'männer',
  'mit',
  'schutzhelmen',
  'bedienen',
  'ein',
  'antriebsradsystem'],
 ['ein',
  'kleines',
  'mädchen',
  'klettert',
  'in',
  'ein',
  'spielhaus',
  'aus',
  'holz'],
 ['ein',
  'mann',
  'in',
  'einem',
  'blauen',
  'hemd',
  'steht',
  'auf',
  'einer',
  'leiter',
  'und',
  'putzt',
  'ein',
  'fenster'],
 ['zwei', 'männer', 'stehen', 'am', 'herd', 'und', 'bereiten', 'essen', 'zu']]

Build a vocab dictionary

In [59]:
from collections import Counter
def build_dictionary(tokens,min_freq=2):
  counter=Counter()
  for text in tokens:
    counter.update(text)
  vocab={"<unk>": 0, "<pad>": 1, "<sos>": 2, "<eos>": 3}
  for word,freq in counter.items():
    if freq>=min_freq:
      vocab[word]=len(vocab)# mapping it to the current size of the 'vocab dict'
  itos={}#index_to_string
  for string,index in vocab.items():
    itos[index]=string
  return vocab,itos

In [60]:
en_vocab,en_itos=build_dictionary(en_tokens)
de_vocab,de_itos=build_dictionary(de_tokens)

In [74]:
en_vocab['beds']

944

In [73]:
de_vocab['dahinter']

941

In [63]:
en_itos[7]

'males'

In [64]:
de_itos[7]

'männer'

In [65]:
def mapping(vocab,text):
  mapped_lists=[]
  default="<unk>"
  for word in text:
    x=vocab.get(word,default)
    mapped_lists.append(x)
  return mapped_lists


In [66]:
mapped_en_texts=[]
for text in en_tokens:
  en_mapped_lists=mapping(en_vocab,text)
  mapped_en_texts.append(en_mapped_lists)
mapped_de_texts=[]
for text in de_tokens:
  de_mapped_lists=mapping(de_vocab,text)
  mapped_de_texts.append(de_mapped_lists)



In [67]:
len(mapped_en_texts)

29000

In [68]:
len(mapped_de_texts)

29000

In [69]:
mapped_en_texts[:5]

[[4, 5, 6, 7, 8, 9, 10, 11, 12],
 [13, 14, 15, 16, 17, 8, 18, 19, 20, 21, 22],
 [19, 23, 24, 25, 26, 19, 27, 28],
 [19, 29, 15, 19, 30, 31, 32, 33, 34, 19, 35, 36, 19, 37],
 [4, 14, 8, 38, 39, 40, 41, 42]]

In [70]:
mapped_de_texts[:5]

[[4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
 [16, 7, 17, 18, 19, 20, '<unk>'],
 [20, 21, 22, 23, 11, 20, 24, 25, 26],
 [20, 27, 11, 28, 29, 30, 31, 32, 33, 34, 35, 36, 20, 37],
 [4, 7, 38, 39, 40, 35, 41, 42, 43]]

In [71]:
import torch
import torch.nn as nn
import torch.optim as optim


In [72]:
class Vanilla_RNN(nn.Module):
  def __init__(self,embed_size_en,vocab_size_en,hidden_size_en,hidden_size_de,vocab_size_de,embed_size_de):
    super(Vanilla_RNN,self).__init__()
    self.rnn_encode=nn.RNN(embed_size_en,hidden_size_en,batch_first=True,num_layers=1)
    self.embed_size_en=nn.Embedding(vocab_size_en,embed_size_en)
    self.rnn_decode=nn.RNN(embed_size_de,hidden_size_de,batch_first=True,num_layers=1)
    self.embed_size_de=nn.Embedding(vocab_size_de,embed_size_de)

    self.output=nn.Linear(hidden_size_de, vocab_size_de)
  def forward(self,x_en,x_de,h0=None):
    e_en=self.embed_size_en(x_en)
    out,hn=self.rnn_encode(e_en,h0)
    e_de=self.embed_size_de(x_de)
    out,hn_de=self.rnn_decode(e_de.unsqueeze(0), hn)
    logits=self.output(out.squeeze(0))
    return logits

